# Optional: stitch images acquired with different beamstops

This notebook loads raw images and their per-image masks. Counts are normalized by exposure time, optional bounded cross-correlation registration and linear/polynomial calibration are applied, and equal-exposure images are averaged wherever they overlap. The longest exposure is used first; shorter groups fill only missing pixels. The stitched mask is the intersection (logical AND) of the aligned input masks: only pixels covered in every input remain masked and `NaN`. Run one of the mask notebooks for every beamstop/image ID first.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

BASEFOLDER = Path.cwd().resolve()
ROOT = BASEFOLDER
sys.path.insert(0, str(ROOT))
from library.beamstop_stitching import stitch_exposures
from library.data_loading import SextantsNexusLoader
from library.mask_store import MaskStore

In [ ]:
# Edit this block. Order does not matter; exposure times determine priority.
IMAGE_IDS = [101, 102, 103]
RAW_FOLDER = Path('/home/experiences/sextants/com-sextants/ruche/sextants-soleil/com-sextants/COMET_20260902_Cocoons_Laser/')
RAW_FOLDER = Path('../COMET_20260902_Cocoons_Laser_raw/')
loader = SextantsNexusLoader(RAW_FOLDER)
mask_store = MaskStore(ROOT / 'processed' / 'mask_pixels')
OUTPUT = ROOT / 'processed' / 'stitched' / 'stitched_101_102_103.npz'

REGISTER = False       # use cross-correlation before intensity fitting
MAX_SHIFT = 10.0       # maximum shift in pixels along either axis
FIT_DEGREE = None      # None: exposure only; 1: linear; 2+: nonlinear polynomial
FIT_PERCENTILES = (2, 98)  # exclude intensity extremes from the fit

In [ ]:
frames = [loader.load(image_id) for image_id in IMAGE_IDS]
masks = [mask_store.load(frame.image_id, frame.image.shape) for frame in frames]
result = stitch_exposures(
    frames, masks, register=REGISTER, max_shift=MAX_SHIFT,
    fit_degree=FIT_DEGREE, fit_percentiles=FIT_PERCENTILES,
)
energies = np.asarray([frame.metadata.get('energy_eV', np.nan) for frame in frames], dtype=float)
if not np.all(np.isfinite(energies)):
    raise ValueError('Every stitched input must contain photon-energy metadata')
energy_eV = float(np.mean(energies))
print('Priority:', result.ordered_ids)
print('Reference exposure:', result.reference_exposure)
print('Pixels missing in every input:', int(result.missing_mask.sum()))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(result.image, cmap='gray')
axes[0].set_title('stitched image')
axes[1].imshow(result.source_exposure, cmap='viridis')
axes[1].set_title('source exposure')
axes[2].imshow(result.missing_mask, cmap='gray')
axes[2].set_title('still missing')
plt.tight_layout()

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    OUTPUT, image=result.image, mask_pixel=result.missing_mask.astype(np.uint8),
    source_count=result.source_count, source_exposure=result.source_exposure,
    reference_exposure=result.reference_exposure,
    ordered_ids=np.asarray(result.ordered_ids), energy_eV=energy_eV,
)
print('Saved:', OUTPUT)

In [ ]:
# Acquisition ID summary
print("im_ids:", globals().get("IMAGE_IDS", globals().get("SCAN_IDS", None)))
print("dark_ids:", globals().get("DARK_IDS", None))